In [35]:
pip install anthropic pandas scikit-learn numbers-parser

Note: you may need to restart the kernel to use updated packages.


In [36]:
# Imports
import os
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from anthropic import Anthropic
from numbers_parser import Document
from sklearn.metrics import cohen_kappa_score, confusion_matrix

In [ ]:
os.environ["ANTHROPIC_API_KEY"] = "API şifremi sildim"

client = Anthropic()
print("Client hazır")

Client hazır


In [38]:

CEMRE_FILE = "/Users/mehmetbagdinli/Desktop/deprem/annotation_set_cemre.numbers"
MEHMET_FILE = "/Users/mehmetbagdinli/Desktop/deprem/annotation_set_mehmet.numbers"


OUTPUT_DIR = Path("/Users/mehmetbagdinli/Desktop/deprem/llm_annotation_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Output klasörü: {OUTPUT_DIR.absolute()}")

Output klasörü: /Users/mehmetbagdinli/Desktop/deprem/llm_annotation_output


In [39]:
def load_annotator(path):
    doc = Document(path)
    table = doc.sheets[0].tables[0]
    rows = table.rows(values_only=True)
    return pd.DataFrame(rows[1:], columns=rows[0])

frames = ['technical', 'political', 'development', 'sustainability']

df_c = load_annotator(CEMRE_FILE)
df_m = load_annotator(MEHMET_FILE)

df_c_done = df_c.dropna(subset=frames).copy()
df_m_done = df_m.dropna(subset=frames).copy()

print(f"annot_1: {len(df_c)} , {len(df_c_done)} labeled")
print(f"annot_2: {len(df_m)} , {len(df_m_done)} labeled")

annot_1: 300 , 99 labeled
annot_2: 300 , 99 labeled


In [40]:
# Per-frame agreement-based gold
gold = df_c_done[['doc_id','source','source_type','province','period','date','text']].copy()
m_indexed = df_m_done.set_index('doc_id')

agreement_summary = {}
for f in frames:
    c_vals = df_c_done[f].astype(int).values
    m_vals = m_indexed.loc[df_c_done['doc_id'], f].astype(int).values
    
    agreed = (c_vals == m_vals)
    
    # Gold: agreement → exact, disagree → min (muhafazakar default)
    gold[f] = np.where(agreed, c_vals, np.minimum(c_vals, m_vals))
    
    # Metadata kolonları
    gold[f'{f}_c'] = c_vals
    gold[f'{f}_m'] = m_vals
    gold[f'{f}_agreed'] = agreed.astype(int)
    gold[f'{f}_diff'] = np.abs(c_vals - m_vals)
    
    agreement_summary[f] = {
        'agreed': agreed.sum(),
        'disagreed': (~agreed).sum(),
        'agreement_rate': agreed.mean(),
        'disagreed_by_2plus': (np.abs(c_vals - m_vals) >= 2).sum()
    }

print("Per-frame agreement özeti (n=99):")
print(f"{'Frame':<16}{'Agreed':<10}{'Disagreed':<12}{'%Agree':<10}{'≥2 fark'}")
print("-" * 55)
for f, s in agreement_summary.items():
    print(f"{f:<16}{s['agreed']:<10}{s['disagreed']:<12}{s['agreement_rate']:<10.1%}{s['disagreed_by_2plus']}")

gold.to_csv(OUTPUT_DIR / 'gold_99.csv', index=False)
print(f"\n✓ Saved: gold_99.csv")

Per-frame agreement özeti (n=99):
Frame           Agreed    Disagreed   %Agree    ≥2 fark
-------------------------------------------------------
technical       88        11          88.9%     11
political       87        12          87.9%     12
development     85        14          85.9%     1
sustainability  99        0           100.0%    0

✓ Saved: gold_99.csv


In [41]:
# To-annotate dataset (201 doc)
remaining = df_c[df_c[frames].isna().any(axis=1)].copy()
remaining = remaining[['doc_id','source','source_type','province','period','date','text']]
remaining.to_csv(OUTPUT_DIR / 'to_annotate_201.csv', index=False)
print(f"Annotate edilecek: {len(remaining)} doc")

Annotate edilecek: 201 doc


In [42]:

all_agreed = (gold[[f'{f}_agreed' for f in frames]].sum(axis=1) == 4)
candidate_pool = gold[all_agreed].copy()
print(f"Tüm 4 frame'de agreement: {all_agreed.sum()} doc (few-shot pool)")


chosen_ids = [
    'hatay_bld_0050',     # all zeros (baseline)
    'adiyaman_val_028',   # P=1 only
    'csb_0523',           # P=2 only
    'kmaras_val_201',     # D=3 only
    'kmaras_bld_0549',    # T=2, P=2, D=1
    'csb_0820',           # T=3, P=1, D=2
    'csb_0354',           # T=3, P=3, D=2
    'hatay_bld_0167',     # T=1, P=1, D=1, S=1 (sustainability!)
]


in_pool = candidate_pool['doc_id'].tolist()
for did in chosen_ids:
    status = "✓" if did in in_pool else "⚠ DİSAGREEMENT VAR"
    print(f"  {did}: {status}")

fewshot_rows = []
for doc_id in chosen_ids:
    row = gold[gold['doc_id']==doc_id].iloc[0]
    fewshot_rows.append({
        'doc_id': doc_id,
        'text': row['text'],
        'technical': int(row['technical']),
        'political': int(row['political']),
        'development': int(row['development']),
        'sustainability': int(row['sustainability']),
    })

fewshot = pd.DataFrame(fewshot_rows)
fewshot.to_csv(OUTPUT_DIR / 'fewshot_8.csv', index=False)

print("\nFew-shot örnekleri:")
for _, r in fewshot.iterrows():
    print(f"  {r['doc_id']:25s} T={r['technical']} P={r['political']} D={r['development']} S={r['sustainability']}")

Tüm 4 frame'de agreement: 67 doc (few-shot pool)
  hatay_bld_0050: ✓
  adiyaman_val_028: ✓
  csb_0523: ✓
  kmaras_val_201: ✓
  kmaras_bld_0549: ✓
  csb_0820: ✓
  csb_0354: ✓
  hatay_bld_0167: ✓

Few-shot örnekleri:
  hatay_bld_0050            T=0 P=0 D=0 S=0
  adiyaman_val_028          T=0 P=1 D=0 S=0
  csb_0523                  T=0 P=2 D=0 S=0
  kmaras_val_201            T=0 P=0 D=3 S=0
  kmaras_bld_0549           T=2 P=2 D=1 S=0
  csb_0820                  T=3 P=1 D=2 S=0
  csb_0354                  T=3 P=3 D=2 S=0
  hatay_bld_0167            T=1 P=1 D=3 S=1


In [43]:
CODEBOOK = """\
4 frame'i 0-3 Likert ölçeğinde değerlendir. Her frame BAĞIMSIZ değerlendirilir;
bir metin birden fazla frame'den yüksek puan alabilir.

FRAME 1 — TECHNICAL (Teknik Dayanıklılık)
Tanım: Deprem/yapı dayanıklılığını mühendislik, yönetmelik, denetim, malzeme,
zemin etüdü, hasar tespiti gibi teknik-bürokratik kavramlar üzerinden çerçeveleyen söylem.
Göstergeler: "yapı denetimi", "zemin etüdü", "yönetmelik", "kentsel dönüşüm" (mühendislik
bağlamında), "rezerv alan", "güçlendirme", "AFAD raporu", sayısal/teknik veri.
0=Yok, 1=İma (1-2 cümle), 2=Belirgin (~%30-50), 3=Baskın (omurga)

FRAME 2 — POLITICAL (Politik Mobilizasyon)
Tanım: Deprem kurtarma/yeniden yapılanmayı seçim kazanımı, partici rekabet, liderlik
meşruiyeti, "biz vs onlar", millet söylemi üzerinden çerçeveleyen söylem.
Göstergeler: "milletimize söz verdik", "muhalefet engelledi", "Sayın Cumhurbaşkanımız",
"AK Parti", seçim vaadi, "başardık", "tarihi adım", tören dili.
0=Yok, 1=İma, 2=Belirgin, 3=Baskın (miting tonu)

FRAME 3 — DEVELOPMENT (Kalkınma ve Ekonomik Dönüşüm)
Tanım: Depremi bir kalkınma fırsatı, ekonomik yatırım, altyapı modernizasyonu veya
bölgesel büyüme vektörü olarak çerçeveleyen söylem.
Göstergeler: "yeni şehir", "modern altyapı", "yatırım", "TOKİ konutları",
"tamamlanan yatırım miktarı", "OSB", "istihdam", proje vurgusu.
0=Yok, 1=İma, 2=Belirgin (yatırım rakamları somut), 3=Baskın (kalkınma vizyonu)

FRAME 4 — SUSTAINABILITY (Sürdürülebilirlik ve Çevre)
Tanım: Yeniden yapılanmayı iklim direnci, çevre dostu yapı, yeşil altyapı, ekolojik
şehir kavramlarıyla çerçeveleyen söylem. Bu corpus'ta NADİR.
Göstergeler: "yeşil bina", "iklim dirençli", "ekolojik", "karbon ayak izi",
"yenilenebilir enerji", "güneş paneli", "enerji verimliliği".
DİKKAT: "modern, konforlu konutlar" ≠ sürdürülebilir. Çevresel sözcük olmadan 0.
0=Yok, 1=İma, 2=Belirgin, 3=Baskın

KARAR REHBERİ:
1. Frame'in göstergeleri VAR MI? Hayır → 0. Evet:
2. Metnin MERKEZİNDE Mİ? Kıyı (1-2 cümle) → 1. Merkez:
3. Metnin OMURGASI MI? Hayır (~%30-50) → 2. Evet (başlık+giriş+sonuç) → 3
"""

MAX_TEXT_CHARS = 8000

def build_fewshot_block(fewshot_df):
    blocks = []
    for _, r in fewshot_df.iterrows():
        blocks.append(
            f"--- ÖRNEK ---\n"
            f"METİN:\n{r['text']}\n\n"
            f"ETİKET (JSON):\n"
            f'{{"technical": {r["technical"]}, '
            f'"political": {r["political"]}, '
            f'"development": {r["development"]}, '
            f'"sustainability": {r["sustainability"]}}}'
        )
    return "\n\n".join(blocks)

def build_prompt(text, fewshot_block):
    text = text[:MAX_TEXT_CHARS]
    return (
        f"{CODEBOOK}\n\n"
        f"==== ANNOTATE EDİLMİŞ ÖRNEKLER ====\n\n"
        f"{fewshot_block}\n\n"
        f"==== ŞİMDİ ANNOTATE ET ====\n\n"
        f"METİN:\n{text}\n\n"
        f"Sadece JSON döndür, başka hiçbir şey yazma. Format:\n"
        f'{{"technical": <0-3>, "political": <0-3>, "development": <0-3>, "sustainability": <0-3>}}'
    )

fewshot_block = build_fewshot_block(fewshot)
print(f"Few-shot block: {len(fewshot_block)} karakter")

Few-shot block: 13556 karakter


In [44]:
MODEL = "claude-haiku-4-5-20251001"  


TEMPERATURE = 0
MAX_TOKENS = 200

def parse_response(content):
    s = content.strip()
    if s.startswith("```"):
        s = s.split("```")[1]
        if s.startswith("json"):
            s = s[4:]
        s = s.strip()
    start = s.find("{")
    end = s.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"JSON yok: {content[:200]}")
    obj = json.loads(s[start:end+1])
    out = {}
    for k in ["technical", "political", "development", "sustainability"]:
        v = int(obj[k])
        assert 0 <= v <= 3, f"{k}={v} aralık dışı"
        out[k] = v
    return out

def annotate_one(text, fewshot_block):
    prompt = build_prompt(text, fewshot_block)
    resp = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        messages=[{"role": "user", "content": prompt}],
    )
    content = resp.content[0].text
    return parse_response(content), content

print(f"Model: {MODEL}")

Model: claude-haiku-4-5-20251001


In [45]:
test_doc = remaining.iloc[0]
print(f"Test doc: {test_doc['doc_id']}")
print(f"Önizleme: {test_doc['text'][:200]}...\n")

labels, raw = annotate_one(test_doc['text'], fewshot_block)
print(f"Raw response: {raw}")
print(f"Parsed: {labels}")

Test doc: kmaras_val_406
Önizleme: Valimiz Mükerrem Ünlüer, Kahramanmaraş Sütçü İmam Üniversitesi Sezai Karakoç Konferans Salonu’nda düzenlenen “Yazılışının 950. Yılında Uluslararası Dîvânu Lugâti't-Türk” sempozyumuna katıldı.

“UNESCO...

Raw response: ```json
{"technical": 0, "political": 1, "development": 1, "sustainability": 0}
```
Parsed: {'technical': 0, 'political': 1, 'development': 1, 'sustainability': 0}


In [46]:
fewshot_ids = set(fewshot['doc_id'])
val_set = gold[~gold['doc_id'].isin(fewshot_ids)].copy()
print(f"Validation seti: {len(val_set)} doc")

results_val = []
errors_val = []
t0 = time.time()
for i, (_, row) in enumerate(val_set.iterrows()):
    try:
        labels, _ = annotate_one(row['text'], fewshot_block)
        results_val.append({'doc_id': row['doc_id'], **labels})
    except Exception as e:
        errors_val.append({'doc_id': row['doc_id'], 'error': str(e)})
        print(f"  HATA {row['doc_id']}: {e}")
    
    if (i+1) % 10 == 0:
        elapsed = time.time() - t0
        rate = (i+1) / elapsed
        eta = (len(val_set) - i - 1) / rate
        print(f"  [{i+1}/{len(val_set)}] {rate:.2f} doc/s, ETA {eta/60:.1f} dk")
    time.sleep(0.3)

llm_on_99 = pd.DataFrame(results_val)
llm_on_99.to_csv(OUTPUT_DIR / 'llm_on_99.csv', index=False)
print(f"\n✓ {len(results_val)} başarılı, {len(errors_val)} hata")

Validation seti: 91 docThe history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.

  [10/91] 0.22 doc/s, ETA 6.2 dk
  [20/91] 0.14 doc/s, ETA 8.6 dk
  [30/91] 0.12 doc/s, ETA 8.8 dk
  [40/91] 0.11 doc/s, ETA 7.9 dk
  [50/91] 0.10 doc/s, ETA 6.6 dk
  [60/91] 0.10 doc/s, ETA 5.1 dk
  [70/91] 0.10 doc/s, ETA 3.5 dk
  [80/91] 0.10 doc/s, ETA 1.9 dk
  [90/91] 0.10 doc/s, ETA 0.2 dk

✓ 91 başarılı, 0 hata


In [47]:
# Üçlü merge
m_data = (gold[['doc_id'] + frames + [f'{f}_c' for f in frames] 
              + [f'{f}_m' for f in frames] + [f'{f}_agreed' for f in frames]]
     .merge(llm_on_99[['doc_id'] + frames].rename(columns={f: f+'_l' for f in frames}), on='doc_id'))

print(f"Üçlü match: {len(m_data)} doc\n")

print("=== AGREEMENT-ONLY VALIDATION (frame başına ayrı subset) ===")
print(f"{'Frame':<16}{'N agreed':<12}{'LLM=human':<12}{'±1':<10}{'κ':<10}{'κ_quad'}")
print("-" * 75)

results_per_frame = {}
for f in frames:
    # Sadece insanların anlaştığı subset
    agreed_mask = m_data[f'{f}_agreed'] == 1
    n_agreed = agreed_mask.sum()
    
    if n_agreed < 2:
        print(f"{f:<16}{n_agreed:<12}(yetersiz veri)")
        continue
    
    human = m_data.loc[agreed_mask, f'{f}_c'].astype(int)  # = m de aynı
    llm = m_data.loc[agreed_mask, f'{f}_l'].astype(int)
    
    exact = (human == llm).mean()
    within1 = (abs(human - llm) <= 1).mean()
    try:
        k = cohen_kappa_score(human, llm)
        kq = cohen_kappa_score(human, llm, weights='quadratic')
    except:
        k, kq = float('nan'), float('nan')
    
    results_per_frame[f] = {'n': n_agreed, 'exact': exact, 'kappa': k, 'kappa_q': kq}
    print(f"{f:<16}{n_agreed:<12}{exact:<12.2%}{within1:<10.2%}{k:<10.3f}{kq:.3f}")

print("\n=== KARAR ===")
for f, r in results_per_frame.items():
    if r['kappa'] >= 0.7:
        v = "✓ LLM güvenilir, 3. annotator olarak kullan"
    elif r['kappa'] >= 0.5:
        v = "△ LLM pre-annotation, manuel kontrol şart"
    else:
        v = "✗ LLM güvenilir değil — manuel devam"
    print(f"  {f:<16} κ={r['kappa']:.3f}  {v}")

Üçlü match: 91 doc

=== AGREEMENT-ONLY VALIDATION (frame başına ayrı subset) ===
Frame           N agreed    LLM=human   ±1        κ         κ_quad
---------------------------------------------------------------------------
technical       80          37.50%      90.00%    0.149     0.587
political       79          53.16%      96.20%    0.362     0.756
development     77          45.45%      88.31%    0.250     0.489
sustainability  91          96.70%      98.90%    0.715     0.886

=== KARAR ===
  technical        κ=0.149  ✗ LLM güvenilir değil — manuel devam
  political        κ=0.362  ✗ LLM güvenilir değil — manuel devam
  development      κ=0.250  ✗ LLM güvenilir değil — manuel devam
  sustainability   κ=0.715  ✓ LLM güvenilir, 3. annotator olarak kullan


In [48]:

print("=== Sistematik kayma (LLM mean - human mean) ===")
for f in frames:
    agreed_mask = m_data[f'{f}_agreed'] == 1
    h = m_data.loc[agreed_mask, f'{f}_c'].astype(int)
    l = m_data.loc[agreed_mask, f'{f}_l'].astype(int)
    delta = l.mean() - h.mean()
    direction = "↑ LLM YÜKSEK puan veriyor" if delta > 0.2 else ("↓ LLM DÜŞÜK puan veriyor" if delta < -0.2 else "≈ kayma yok")
    print(f"  {f:<16} human={h.mean():.2f}  llm={l.mean():.2f}  Δ={delta:+.2f}  {direction}")

print("\n=== Confusion matrices (rows=human, cols=LLM) ===")
for f in frames:
    agreed_mask = m_data[f'{f}_agreed'] == 1
    h = m_data.loc[agreed_mask, f'{f}_c'].astype(int)
    l = m_data.loc[agreed_mask, f'{f}_l'].astype(int)
    cm = confusion_matrix(h, l, labels=[0,1,2,3])
    print(f"\n{f.upper()}:")
    print("       LLM=0  LLM=1  LLM=2  LLM=3")
    for i, row in enumerate(cm):
        print(f"H={i}    {row[0]:5d}  {row[1]:5d}  {row[2]:5d}  {row[3]:5d}")

=== Sistematik kayma (LLM mean - human mean) ===
  technical        human=0.69  llm=1.26  Δ=+0.57  ↑ LLM YÜKSEK puan veriyor
  political        human=1.73  llm=1.91  Δ=+0.18  ≈ kayma yok
  development      human=1.35  llm=1.60  Δ=+0.25  ↑ LLM YÜKSEK puan veriyor
  sustainability   human=0.13  llm=0.13  Δ=+0.00  ≈ kayma yok

=== Confusion matrices (rows=human, cols=LLM) ===

TECHNICAL:
       LLM=0  LLM=1  LLM=2  LLM=3
H=0       17     24      6      0
H=1        2      6     10      1
H=2        0      1      1      4
H=3        0      1      1      6

POLITICAL:
       LLM=0  LLM=1  LLM=2  LLM=3
H=0        7      8      1      0
H=1        2      4     11      1
H=2        0      4      8      4
H=3        0      1      5     23

DEVELOPMENT:
       LLM=0  LLM=1  LLM=2  LLM=3
H=0       10      7      1      1
H=1        0      8     11      2
H=2        3      2     14      9
H=3        1      1      4      3

SUSTAINABILITY:
       LLM=0  LLM=1  LLM=2  LLM=3
H=0       85      0      

In [49]:

MODEL = "claude-sonnet-4-5"

quick_test = val_set.head(20)
sonnet_results = []
for _, row in quick_test.iterrows():
    try:
        labels, _ = annotate_one(row['text'], fewshot_block)
        sonnet_results.append({'doc_id': row['doc_id'], **labels})
    except Exception as e:
        print(f"HATA {row['doc_id']}: {e}")
    time.sleep(0.3)

sonnet_df = pd.DataFrame(sonnet_results)

# 20 doc üzerinde Sonnet vs Haiku karşılaştırması
compare = (m_data[m_data['doc_id'].isin(sonnet_df['doc_id'])]
           [['doc_id'] + [f'{f}_c' for f in frames] + [f'{f}_agreed' for f in frames]]
           .merge(sonnet_df.rename(columns={f: f+'_s' for f in frames}), on='doc_id'))

print("Sonnet vs Haiku (20 doc subset):")
for f in frames:
    agreed = compare[f'{f}_agreed'] == 1
    if agreed.sum() < 5: continue
    h = compare.loc[agreed, f'{f}_c'].astype(int)
    s = compare.loc[agreed, f'{f}_s'].astype(int)
    exact = (h==s).mean()
    delta = s.mean() - h.mean()
    print(f"  {f:<16} exact={exact:.0%}  Δ_mean={delta:+.2f}  (Haiku Δ vardı: T+0.57, P+0.18, D+0.25, S+0.00)")

Sonnet vs Haiku (20 doc subset):
  technical        exact=53%  Δ_mean=+0.00  (Haiku Δ vardı: T+0.57, P+0.18, D+0.25, S+0.00)
  political        exact=37%  Δ_mean=-0.47  (Haiku Δ vardı: T+0.57, P+0.18, D+0.25, S+0.00)
  development      exact=47%  Δ_mean=+0.12  (Haiku Δ vardı: T+0.57, P+0.18, D+0.25, S+0.00)
  sustainability   exact=95%  Δ_mean=-0.05  (Haiku Δ vardı: T+0.57, P+0.18, D+0.25, S+0.00)


In [50]:

import os

out_path = OUTPUT_DIR / 'llm_on_201.csv'
if out_path.exists():
    os.remove(out_path)
    print("✓ Eski llm_on_201.csv silindi")
else:
    print("Zaten yok, temiz başlanacak")


val_path = OUTPUT_DIR / 'llm_on_99.csv'
if val_path.exists():
    os.remove(val_path)
    print("✓ Eski llm_on_99.csv da silindi (Sonnet validation için temiz)")


print(f"\nKullanılacak model: {MODEL}")
assert "sonnet" in MODEL.lower(), "DİKKAT: Model hala Sonnet değil! Hücre 8'e dön ve değiştir."
print("✓ Sonnet hazır, hücre 12'yi çalıştırabilirsin")

Zaten yok, temiz başlanacak
✓ Eski llm_on_99.csv da silindi (Sonnet validation için temiz)

Kullanılacak model: claude-sonnet-4-5
✓ Sonnet hazır, hücre 12'yi çalıştırabilirsin


In [51]:
MODEL = "claude-sonnet-4-5"   


TEMPERATURE = 0
MAX_TOKENS = 200

def parse_response(content):
    s = content.strip()
    if s.startswith("```"):
        s = s.split("```")[1]
        if s.startswith("json"):
            s = s[4:]
        s = s.strip()
    start = s.find("{")
    end = s.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"JSON yok: {content[:200]}")
    obj = json.loads(s[start:end+1])
    out = {}
    for k in ["technical", "political", "development", "sustainability"]:
        v = int(obj[k])
        assert 0 <= v <= 3, f"{k}={v} aralık dışı"
        out[k] = v
    return out

def annotate_one(text, fewshot_block):
    prompt = build_prompt(text, fewshot_block)
    resp = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        messages=[{"role": "user", "content": prompt}],
    )
    content = resp.content[0].text
    return parse_response(content), content

print(f"Model: {MODEL}")

Model: claude-sonnet-4-5


In [52]:
results_201 = []
errors_201 = []
out_path = OUTPUT_DIR / 'llm_on_201.csv'

# Resume
done_ids = set()
if out_path.exists():
    existing = pd.read_csv(out_path)
    done_ids = set(existing['doc_id'])
    results_201 = existing.to_dict('records')
    print(f"Resume: {len(done_ids)} zaten yapılmış")

t0 = time.time()
for i, (_, row) in enumerate(remaining.iterrows()):
    if row['doc_id'] in done_ids:
        continue
    try:
        labels, _ = annotate_one(row['text'], fewshot_block)
        rec = {'doc_id': row['doc_id'], **labels}
        results_201.append(rec)
        pd.DataFrame(results_201).to_csv(out_path, index=False)
    except Exception as e:
        errors_201.append({'doc_id': row['doc_id'], 'error': str(e)})
        print(f"  HATA {row['doc_id']}: {e}")
    
    if (i+1) % 10 == 0:
        elapsed = time.time() - t0
        rate = (i+1) / elapsed
        eta = (len(remaining) - i - 1) / rate
        print(f"  [{i+1}/{len(remaining)}] {rate:.2f} doc/s, ETA {eta/60:.1f} dk")
    time.sleep(0.3)

print(f"\n✓ {len(results_201)} sonuç, {len(errors_201)} hata")

  [10/201] 0.09 doc/s, ETA 34.9 dk
  [20/201] 0.07 doc/s, ETA 43.2 dk
  [30/201] 0.06 doc/s, ETA 44.7 dk
  [40/201] 0.06 doc/s, ETA 44.6 dk
  [50/201] 0.06 doc/s, ETA 42.6 dk
  [60/201] 0.06 doc/s, ETA 40.4 dk
  [70/201] 0.06 doc/s, ETA 38.4 dk
  [80/201] 0.06 doc/s, ETA 35.7 dk
  [90/201] 0.06 doc/s, ETA 33.1 dk
  [100/201] 0.06 doc/s, ETA 30.1 dk
  [110/201] 0.06 doc/s, ETA 27.2 dk
  [120/201] 0.06 doc/s, ETA 24.3 dk
  [130/201] 0.06 doc/s, ETA 21.2 dk
  [140/201] 0.05 doc/s, ETA 20.6 dk
  [150/201] 0.05 doc/s, ETA 17.1 dk
  [160/201] 0.05 doc/s, ETA 13.6 dk
  [170/201] 0.05 doc/s, ETA 10.2 dk
  [180/201] 0.05 doc/s, ETA 6.9 dk
  [190/201] 0.05 doc/s, ETA 3.6 dk
  [200/201] 0.05 doc/s, ETA 0.3 dk

✓ 201 sonuç, 0 hata


In [54]:
# Sonnet'i 91 gold doc üzerinde çalıştır ve κ hesapla
val_path = OUTPUT_DIR / 'llm_on_99.csv'

# Eğer eski (Haiku) sonuçlar varsa sil
if val_path.exists():
    os.remove(val_path)
    print("Eski validation sonuçları silindi, Sonnet ile baştan")

# Few-shot dışındaki gold'lar
fewshot_ids = set(fewshot['doc_id'])
val_set = gold[~gold['doc_id'].isin(fewshot_ids)].copy()
print(f"Validation seti: {len(val_set)} doc, model: {MODEL}\n")

results_val = []
errors_val = []
t0 = time.time()
for i, (_, row) in enumerate(val_set.iterrows()):
    try:
        labels, _ = annotate_one(row['text'], fewshot_block)
        results_val.append({'doc_id': row['doc_id'], **labels})
    except Exception as e:
        errors_val.append({'doc_id': row['doc_id'], 'error': str(e)})
        print(f"  HATA {row['doc_id']}: {e}")
    
    if (i+1) % 10 == 0:
        elapsed = time.time() - t0
        rate = (i+1) / elapsed
        eta = (len(val_set) - i - 1) / rate
        print(f"  [{i+1}/{len(val_set)}] {rate:.2f} doc/s, ETA {eta/60:.1f} dk")
    time.sleep(0.3)

llm_on_99 = pd.DataFrame(results_val)
llm_on_99.to_csv(val_path, index=False)
print(f"\n✓ {len(results_val)} başarılı, {len(errors_val)} hata")

Eski validation sonuçları silindi, Sonnet ile baştan
Validation seti: 91 doc, model: claude-sonnet-4-5

  [10/91] 0.05 doc/s, ETA 24.6 dk
  [20/91] 0.06 doc/s, ETA 20.6 dk
  [30/91] 0.06 doc/s, ETA 18.3 dk
  [40/91] 0.06 doc/s, ETA 15.4 dk
  [50/91] 0.06 doc/s, ETA 12.4 dk
  [60/91] 0.05 doc/s, ETA 9.4 dk
  [70/91] 0.05 doc/s, ETA 6.4 dk
  [80/91] 0.06 doc/s, ETA 3.3 dk
  [90/91] 0.05 doc/s, ETA 0.3 dk

✓ 91 başarılı, 0 hata


In [55]:
# Üçlü merge
m_data = (gold[['doc_id'] + frames + [f'{f}_c' for f in frames] 
              + [f'{f}_m' for f in frames] + [f'{f}_agreed' for f in frames]]
     .merge(llm_on_99[['doc_id'] + frames].rename(columns={f: f+'_l' for f in frames}), on='doc_id'))

print(f"=== {MODEL} VALIDATION (agreement-only subset) ===\n")
print(f"{'Frame':<16}{'N':<6}{'Exact':<10}{'±1':<10}{'κ':<10}{'κ_quad':<10}{'Δ_mean'}")
print("-" * 75)

results_summary = {}
for f in frames:
    agreed_mask = m_data[f'{f}_agreed'] == 1
    n = agreed_mask.sum()
    h = m_data.loc[agreed_mask, f'{f}_c'].astype(int)
    l = m_data.loc[agreed_mask, f'{f}_l'].astype(int)
    exact = (h==l).mean()
    within1 = (abs(h-l)<=1).mean()
    k = cohen_kappa_score(h, l)
    kq = cohen_kappa_score(h, l, weights='quadratic')
    delta = l.mean() - h.mean()
    results_summary[f] = {'kappa': k, 'kappa_q': kq, 'exact': exact, 'within1': within1, 'delta': delta}
    print(f"{f:<16}{n:<6}{exact:<10.2%}{within1:<10.2%}{k:<10.3f}{kq:<10.3f}{delta:+.2f}")

print("\n=== KARAR (κ_quad bazlı, ordinal task için doğru metrik) ===")
for f, r in results_summary.items():
    if r['kappa_q'] >= 0.8:
        v = " MÜKEMMEL — LLM 3. annotator olarak güvenle kullan"
    elif r['kappa_q'] >= 0.6:
        v = " İYİ — LLM kullanılabilir, needs_review listesini gözle"
    elif r['kappa_q'] >= 0.4:
        v = " ORTA — LLM pre-annotation, manuel kontrol şart"
    else:
        v = " ZAYIF — LLM güvenilir değil"
    print(f"  {f:<16} κ_quad={r['kappa_q']:.3f}  {v}")

# Confusion matrices
print("\n=== Confusion matrices (rows=human, cols=LLM) ===")
for f in frames:
    agreed_mask = m_data[f'{f}_agreed'] == 1
    h = m_data.loc[agreed_mask, f'{f}_c'].astype(int)
    l = m_data.loc[agreed_mask, f'{f}_l'].astype(int)
    cm = confusion_matrix(h, l, labels=[0,1,2,3])
    print(f"\n{f.upper()}:")
    print("       LLM=0  LLM=1  LLM=2  LLM=3")
    for i, row in enumerate(cm):
        print(f"H={i}    {row[0]:5d}  {row[1]:5d}  {row[2]:5d}  {row[3]:5d}")

=== claude-sonnet-4-5 VALIDATION (agreement-only subset) ===

Frame           N     Exact     ±1        κ         κ_quad    Δ_mean
---------------------------------------------------------------------------
technical       80    48.75%    92.50%    0.253     0.639     +0.36
political       79    49.37%    92.41%    0.303     0.700     +0.25
development     77    46.75%    87.01%    0.255     0.472     +0.13
sustainability  91    94.51%    98.90%    0.528     0.831     -0.02

=== KARAR (κ_quad bazlı, ordinal task için doğru metrik) ===
  technical        κ_quad=0.639   İYİ — LLM kullanılabilir, needs_review listesini gözle
  political        κ_quad=0.700   İYİ — LLM kullanılabilir, needs_review listesini gözle
  development      κ_quad=0.472   ORTA — LLM pre-annotation, manuel kontrol şart
  sustainability   κ_quad=0.831   MÜKEMMEL — LLM 3. annotator olarak güvenle kullan

=== Confusion matrices (rows=human, cols=LLM) ===

TECHNICAL:
       LLM=0  LLM=1  LLM=2  LLM=3
H=0       23     21

In [56]:
llm_201 = pd.read_csv(OUTPUT_DIR / 'llm_on_201.csv').drop_duplicates('doc_id', keep='last')


gold_part = gold.copy()
for f in frames:
    gold_part[f'{f}_source'] = np.where(
        gold_part[f'{f}_agreed'] == 1, 'human_agreement', 'human_disagreement'
    )


llm_meta = df_c[df_c[frames].isna().any(axis=1)][
    ['doc_id','source','source_type','province','period','date','text']]
llm_part = llm_meta.merge(llm_201, on='doc_id', how='left')
for f in frames:
    llm_part[f'{f}_c'] = np.nan
    llm_part[f'{f}_m'] = np.nan
    llm_part[f'{f}_agreed'] = np.nan
    llm_part[f'{f}_diff'] = np.nan
    llm_part[f'{f}_source'] = np.where(llm_part[f].isna(), 'MISSING', 'llm_only')


common_cols = (['doc_id','source','source_type','province','period','date','text']
               + frames
               + [f'{f}_c' for f in frames]
               + [f'{f}_m' for f in frames]
               + [f'{f}_agreed' for f in frames]
               + [f'{f}_diff' for f in frames]
               + [f'{f}_source' for f in frames])

final = pd.concat([gold_part[common_cols], llm_part[common_cols]], ignore_index=True)
final.to_csv(OUTPUT_DIR / 'final_300.csv', index=False)

print(f"=== Final dataset: {len(final)} doc ===\n")
print("Per-frame label source dağılımı:")
for f in frames:
    print(f"\n{f}:")
    print(final[f'{f}_source'].value_counts().to_string())

=== Final dataset: 300 doc ===

Per-frame label source dağılımı:

technical:
technical_source
llm_only              201
human_agreement        88
human_disagreement     11

political:
political_source
llm_only              201
human_agreement        87
human_disagreement     12

development:
development_source
llm_only              201
human_agreement        85
human_disagreement     14

sustainability:
sustainability_source
llm_only           201
human_agreement     99


In [60]:

sus3_docs = final[(final['sustainability_source']=='llm_only') & (final['sustainability']==3)]

print(f"=== Sustainability=3 doc'ları ({len(sus3_docs)} adet) ===")
print("'yeşil bina', 'iklim direnci', 'ekolojik', 'yenilenebilir enerji', 'güneş paneli',")
print("'karbon ayak izi', 'enerji verimliliği' kelimeleri gerçekten geçiyor mu kontrol et.\n")

for _, row in sus3_docs.iterrows():
    print(f"\n{'='*70}")
    print(f"DOC: {row['doc_id']}  (kaynak: {row['source_type']})")
    print(f"LLM tahmin: T={row['technical']}, P={row['political']}, D={row['development']}, S={row['sustainability']}")
    print(f"{'='*70}")
    print(row['text'][:2000])
    if len(row['text']) > 2000:
        print(f"\n[...] (toplam {len(row['text'])} karakter)")

=== Sustainability=3 doc'ları (4 adet) ===
'yeşil bina', 'iklim direnci', 'ekolojik', 'yenilenebilir enerji', 'güneş paneli',
'karbon ayak izi', 'enerji verimliliği' kelimeleri gerçekten geçiyor mu kontrol et.


DOC: csb_0669  (kaynak: bakanlık)
LLM tahmin: T=3, P=1, D=2, S=3
ÇEVRE, ŞEHİRCİLİK VE İKLİM DEĞİŞİKLİĞİ BAKAN YARDIMCISI REFİK TUZCUOĞLU: “KAMU BİNALARINDA YÜZDE 40 ORANINDA ENERJİ TASARRUFU SAĞLADIK”

Çevre, Şehircilik ve İklim Değişikliği Bakan Yardımcısı Refik Tuzcuoğlu, KADEV Projesi kapsamında 565 kamu binasının 318’inin enerji verimliliğine kavuşturulduğunu vurgulayarak, “Bu sayede, yüzde 40 oranında enerji tasarrufu sağladık. Çalışmalarımız devam ediyor. Bu çalışmaları, ülke politikası olarak devam ettireceğiz.” dedi.

Dünya Bankası’nın toplamda 265 milyon dolar tutarındaki kredisi, Hazine ve Maliye Bakanlığı’nın mali garantörlüğünde, Çevre, Şehircilik ve İklim Değişikliği Bakanlığı Yapı İşleri Genel Müdürlüğü’ne bağlı Uluslararası Finans Kaynaklı Sismik Güçlendirme Dair

In [59]:
dev3_docs = final[(final['development_source']=='llm_only') & (final['development']==3)]

print(f"=== Development=3 doc'ları ({len(dev3_docs)} adet) ===")
print("Soru: 'Kalkınma vizyonu METNİN OMURGASI mı?' Sadece yan değiniyorsa → 2 yap\n")

for _, row in dev3_docs.iterrows():
    print(f"\n{'='*70}")
    print(f"DOC: {row['doc_id']}  (kaynak: {row['source_type']})")
    print(f"LLM tahmin: T={row['technical']}, P={row['political']}, D={row['development']}, S={row['sustainability']}")
    print(f"{'='*70}")
    print(row['text'][:600])
    if len(row['text']) > 600:
        print(f"\n[...] (toplam {len(row['text'])} karakter)")

=== Development=3 doc'ları (39 adet) ===
Soru: 'Kalkınma vizyonu METNİN OMURGASI mı?' Sadece yan değiniyorsa → 2 yap


DOC: csb_0291  (kaynak: bakanlık)
LLM tahmin: T=2, P=2, D=3, S=0
DEPREM BÖLGESİNDE YENİ DÖNEM BAŞLIYOR

Deprem bölgesinin ihya ve inşasında yeni bir dönem başlıyor. Çevre, Şehircilik ve İklim Değişikliği Bakanlığı, 17 Temmuz’da başlatacağı ‘Yerinde Dönüşüm’ projesiyle 11 şehri yeniden ayağa kaldırıyor. Çevre, Şehircilik ve İklim Değişikliği Bakanı Mehmet Özhaseki, Yerinde Dönüşüm projesiyle Bakanlığın denetiminde yerinde dönüştürülecek konutlara 500 bin TL hibe, 500 bin TL kredi; işyerlerine ise 250 bin TL hibe, 250 bin TL de kredi desteği verileceğini söyledi. Krediler, 2 yılı ödemesiz 10 yıl vade ve sıfır faizle kullandırılacak. Başvurular ‘e-Devlet’ üzer

[...] (toplam 5481 karakter)

DOC: tccb_051  (kaynak: cumhurbaskanligi)
LLM tahmin: T=2, P=3, D=3, S=0
Hasret Gidermemize, hasbihal etmemize, kucaklaşmamıza vesile olan herkese teşekkür ediyorum. Bir kelam-ı kibar 

In [61]:
manual_fixes = {
    # Sustainability 
    'csb_0669': {'sustainability': 2},
    
    # Developmen
    'kmaras_bld_0070': {'development': 2},
    'kmaras_bld_2179': {'development': 2},
    'csb_0171': {'development': 2},
    'hatay_bld_0026': {'development': 2},
    'hatay_bld_0206': {'development': 2},
    'kmaras_bld_0458': {'development': 2},
    'kmaras_bld_1226': {'development': 2},
    'kmaras_bld_0238': {'development': 2},
    'hatay_bld_0023': {'development': 2},
    'kmaras_bld_2051': {'development': 2},
    'kmaras_bld_0388': {'development': 2},
    'kmaras_bld_0256': {'development': 2},
    'kmaras_bld_0298': {'development': 2},
    'csb_0174': {'development': 2},
    'hatay_bld_0218': {'development': 2},
}

n_fixes = 0
for doc_id, fixes in manual_fixes.items():
    for frame, new_val in fixes.items():
        mask = final['doc_id'] == doc_id
        if mask.sum() == 0:
            print(f"⚠ Bulunamadı: {doc_id}")
            continue
        old_val = int(final.loc[mask, frame].iloc[0])
        final.loc[mask, frame] = new_val
        final.loc[mask, f'{frame}_source'] = 'llm_then_manual'
        print(f"  {doc_id}: {frame} {old_val} → {new_val}")
        n_fixes += 1

final.to_csv(OUTPUT_DIR / 'final_300.csv', index=False)
print(f"\n✓ {n_fixes} düzeltme uygulandı")
print(f"\nGüncellenmiş Development dağılımı:")
print(final['development'].value_counts().sort_index())
print(f"\nGüncellenmiş Sustainability dağılımı:")
print(final['sustainability'].value_counts().sort_index())

  csb_0669: sustainability 3 → 2
  kmaras_bld_0070: development 3 → 2
  kmaras_bld_2179: development 3 → 2
  csb_0171: development 3 → 2
  hatay_bld_0026: development 3 → 2
  hatay_bld_0206: development 3 → 2
  kmaras_bld_0458: development 3 → 2
  kmaras_bld_1226: development 3 → 2
  kmaras_bld_0238: development 3 → 2
  hatay_bld_0023: development 3 → 2
  kmaras_bld_2051: development 3 → 2
  kmaras_bld_0388: development 3 → 2
  kmaras_bld_0256: development 3 → 2
  kmaras_bld_0298: development 3 → 2
  csb_0174: development 3 → 2
  hatay_bld_0218: development 3 → 2

✓ 16 düzeltme uygulandı

Güncellenmiş Development dağılımı:
development
0     69
1     61
2    135
3     35
Name: count, dtype: int64

Güncellenmiş Sustainability dağılımı:
sustainability
0    273
1     15
2      7
3      5
Name: count, dtype: int64
